[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/philippe-amuzuga/Meca-PY_Ed2026/blob/main/python/P06/P06-1_Tableaux_CORRIGE.ipynb)

# Calcul sur tableaux

Chapitre 6 du livre *Python pour l'ingénierie mécanique, industrielle et productique*.

## Cadrage

**Contexte industriel.** Votre bureau d'études doit connaître la distance de chaque nœud d'un maillage à un trou, la contrainte équivalente en mille points d'une pièce, la moyenne de dix mille jauges, et la boucle Python met plusieurs secondes là où le résultat devrait venir d'un trait ? Vous voudriez écrire la formule une fois et l'appliquer à tout un tableau.

**Objectif terminal.** Ce chapitre montre comment, avec NumPy.

**Objectif instrumental.**

- Calculer une grille de valeurs sans boucle par la diffusion de NumPy.
- Réduire un tableau, choisir le type de ses nombres, distinguer vue et copie.
- Empiler des matrices et en tirer les contraintes principales.
- Chronométrer un calcul avant de l'optimiser.

**Critère de résultat.** Chaque exercice atteint son résultat attendu, que des contrôles du notebook vérifient.

**Vérification.** Beaucoup d'exercices donnent une seconde voie : un calcul fait par deux chemins indépendants est un calcul qu'on peut croire.

**Prérequis.** Les chapitres 4 et 5, après une filière scientifique.

## Source de données

Aucun jeu de données : les valeurs viennent de l'énoncé ou sont engendrées par le code, à graine fixée.

## Exercices

| N° | Exercice | Durée indicative |
|:-|:-|-:|
| 6.1 | Grille de distances par diffusion | 15 min |
| 6.2 | Réduction par axe et type `float32` | 15 min |
| 6.3 | Contrainte de von Mises par `eigvalsh` | 20 min |
| 6.4 | Rapport de temps entre boucles et tableau (défi) | 20 min |

Dans le sujet, les cellules marquées `# Votre code :` sont à écrire ; le corrigé les donne résolues. Chaque exercice se termine par des contrôles qui comparent votre résultat à celui du livre.

### En-tête de reproductibilité

Versions des bibliothèques employées : les valeurs du livre sont celles de ces versions, avec des graines fixées.

In [1]:
# En-tête de reproductibilité : versions employées par ce notebook.
import platform
from importlib.metadata import version

print("Python", platform.python_version())
print("numpy", version("numpy"))
print("scikit-learn", version("scikit-learn"))

Python 3.12.0
numpy 2.4.6
scikit-learn 1.9.0


### Préparation du dépôt (Colab)

Ouvert seul dans Colab, ce notebook ne voit pas les dossiers du dépôt. La cellule suivante ne télécharge rien : elle cherche `donnees/` et `outils/` (à deux dossiers au-dessus du notebook, à côté de lui ou dans un clone nommé `Meca-PY_Ed2026`), ajoute `outils/` au chemin de recherche de Python, et, sur Colab, liste ce qui manque. Pour le fournir, ouvrez le panneau des fichiers de Colab, créez les dossiers `donnees` et `outils` à la racine de l'espace de travail, et téléversez-y les fichiers listés, pris dans le dépôt. Ce notebook n'a besoin d'aucun jeu ni d'aucun module : la cellule ne signale rien, mais elle prépare l'import des modules du livre dans vos propres cellules.

In [2]:
# Préparation du dépôt (utile sur Colab) : cette cellule ne télécharge rien.
import os
import sys
from pathlib import Path

JEUX_LUS = []
MODULES_DU_LIVRE = []                          # fichiers de outils/


def chercher(dossier):
    """Le dossier du dépôt cherché depuis le notebook, puis depuis un clone ; None s'il est introuvable."""
    for base in (Path("..", ".."), Path("."), Path("Meca-PY_Ed2026")):
        if (base / dossier).is_dir():
            return base / dossier
    return None


donnees, outils = chercher("donnees"), chercher("outils")
if outils is not None and str(outils) not in sys.path:
    sys.path.insert(0, str(outils))
manque = [f"donnees/{nom}" for nom in JEUX_LUS if donnees is None or not (donnees / nom).is_file()]
manque += [f"outils/{nom}" for nom in MODULES_DU_LIVRE if outils is None or not (outils / nom).is_file()]
if manque and "COLAB_RELEASE_TAG" in os.environ:          # variable posée par Colab
    print("À téléverser dans l'espace de travail de Colab, panneau des fichiers :")
    print("\n".join("  " + nom for nom in manque))


### Fonctions du module du livre (`meca.py`)

Le code ci-dessous reprend, tel qu'il est écrit dans le livre, les fonctions du module employées dans ce notebook ; ce code est fourni tel quel.

In [3]:
import numpy as np


def safe_div(a, b, default=0.0):
    """Division protégée : un dénominateur nul rend `default` au lieu d'une erreur ou d'un infini.

    >>> safe_div(np.array([1.0, 2.0, 3.0]), np.array([0.0, 4.0, 2.0])).tolist()
    [0.0, 0.5, 1.5]
    """
    b_ = np.where(b == 0, np.nan, b)
    out = a / b_
    return np.nan_to_num(out, nan=default, posinf=default, neginf=default)

### Préparation

In [4]:
import math
import time
import timeit
import numpy as np
from sklearn.datasets import load_digits

### Section 6.1 : la diffusion remplace les boucles imbriquées.

In [5]:
# notion: diffusion
x_mm = np.arange(0.0, 101.0, 1.0)              # 101 colonnes, pas de 1 mm
y_mm = np.arange(0.0, 51.0, 1.0)               # 51 lignes
colonne = y_mm.reshape(-1, 1)                  # forme (51, 1) : une valeur par ligne
ligne = x_mm[np.newaxis, :]                    # forme (1, 101) : une valeur par colonne
print("colonne :", colonne.shape, "; ligne :", ligne.shape)
assert colonne.shape == (51, 1) and ligne.shape == (1, 101)
grille_somme = colonne + ligne                 # NumPy étire la colonne et la ligne à la forme (51, 101)
print("colonne + ligne :", grille_somme.shape)
assert grille_somme.shape == (51, 101)
assert (
    grille_somme[3, 5] == y_mm[3] + x_mm[5]
), "chaque case est la somme de sa ligne et de sa colonne"
a = np.ones((3, 4))
b = np.ones((3, 5))

colonne : (51, 1) ; ligne : (1, 101)
colonne + ligne : (51, 101)


In [6]:
try:
    a + b
    message = "aucune erreur"
except ValueError:
    message = "ValueError : formes incompatibles"

print(message)
assert message.startswith("ValueError"), "(3, 4) et (3, 5) ne s'étirent pas l'une sur l'autre"

ValueError : formes incompatibles


## Exercice 6.1 : Grille de distances par diffusion (15 min)

Une plaque de $100 \times 50$ mm est maillée au pas de $1$ mm : $101$ colonnes et $51$ lignes de nœuds. Son trou, de rayon $10$ mm, est centré en $(50~;~25)$ mm. Calculez sans boucle la distance de chaque nœud au centre du trou, puis comptez les nœuds situés strictement dans le trou. Comparez ce nombre à l'aire du disque, en nombre de cases d'un millimètre carré.

**Résultat attendu.** Grille de 51 lignes et 101 colonnes, soit 5 151 nœuds ; 305 nœuds dans le trou, pour une aire de disque de 314,2 mm².

**Seconde voie.** La double boucle sur les 5 151 nœuds, ou `np.meshgrid`, donne la même grille (`np.allclose`).

**Indication.** Les abscisses valent `np.arange(0, 101)`, les ordonnées `np.arange(0, 51)`. On pose `xl = x[np.newaxis, :]` et `yc = y.reshape(-1, 1)` ; `np.sqrt((xl - 50)**2 + (yc - 25)**2)` donne une grille de forme $(51, 101)$. Le masque `distance < 10` est vrai dans le trou, et sa somme compte les nœuds : proche de $\pi\,r^2$.

Les contrôles de la cellule lisent les variables `distance`, `grille_x`, `grille_y`, `n_trou`, `par_boucles` : gardez ces noms.

In [7]:
x0_mm, y0_mm, rayon_mm = 50.0, 25.0, 10.0
# Solution :
distance = np.sqrt((ligne - x0_mm)**2 + (colonne - y0_mm)**2)       # forme (51, 101), sans boucle
dans_le_trou = distance < rayon_mm
n_trou = int(dans_le_trou.sum())
print("forme de la grille :", distance.shape, "; nœuds :", distance.size)
print(f"distance minimale : {distance.min():.1f} mm ; nœuds dans le trou : {n_trou}")
grille_x, grille_y = np.meshgrid(x_mm, y_mm)
# Fin de la solution
assert (
    np.allclose(distance, np.hypot(grille_x - x0_mm, grille_y - y0_mm))
), "même grille par meshgrid"
# Solution :
par_boucles = np.zeros((51, 101))

for i in range(51):
    for j in range(101):
        par_boucles[i, j] = math.sqrt((x_mm[j] - x0_mm)**2 + (y_mm[i] - y0_mm)**2)
# Fin de la solution
assert np.allclose(distance, par_boucles), "même grille que la double boucle"
assert distance.shape == (51, 101) and distance.size == 5151, "51 lignes, 101 colonnes"
assert (
    distance.min() == 0.0 and tuple(np.argwhere(distance == 0.0)[0]) == (25, 50)
), "centre du trou"
assert n_trou == 305, "nœuds du trou attendus : 305 (distance strictement sous 10 mm)"
assert (
    abs(n_trou - math.pi * rayon_mm**2) < 0.05 * math.pi * rayon_mm**2
), "proche de l'aire du disque"
print("Exercice 6.1 : contrôles passés")

forme de la grille : (51, 101) ; nœuds : 5151
distance minimale : 0.0 mm ; nœuds dans le trou : 305
Exercice 6.1 : contrôles passés


### Section 6.2 : les axes, le type et la mémoire fixent le résultat.

In [8]:
rng = np.random.default_rng(2026)
# notion: axes
jauges = rng.normal(100.0, 15.0, size=(10000, 50))      # 10 000 pièces, 50 jauges chacune, en MPa
par_piece = jauges.mean(axis=1)                          # une moyenne par pièce : forme (10000,)
par_jauge = jauges.mean(axis=0)                          # une moyenne par jauge : forme (50,)
centre = jauges - jauges.mean(axis=1, keepdims=True)     # keepdims garde la forme (10000, 1) : la diffusion opère
print("axis=1 :", par_piece.shape, "; axis=0 :", par_jauge.shape, "; keepdims :", jauges.mean(axis=1, keepdims=True).shape)
assert par_piece.shape == (10000,) and par_jauge.shape == (50,)
assert jauges.mean(axis=1, keepdims=True).shape == (10000, 1)
assert np.allclose(centre.mean(axis=1), 0.0), "après centrage, chaque pièce a une moyenne nulle"
assert abs(par_piece.mean() - jauges.mean()) < 1e-9, "la moyenne des moyennes est la moyenne"

axis=1 : (10000,) ; axis=0 : (50,) ; keepdims : (10000, 1)


In [9]:
# notion: dtype
simple = jauges.astype(np.float32)
print("types :", jauges.dtype, "et", simple.dtype, "; octets :", jauges.nbytes, "et", simple.nbytes)
assert jauges.dtype == np.float64 and simple.dtype == np.float32
assert simple.nbytes * 2 == jauges.nbytes, "float32 occupe deux fois moins de mémoire"
# notion: vue-memoire
transposee = jauges.T
copie = jauges.copy()
decoupe = jauges[:10]
print("la transposée est une vue :", transposee.base is jauges, "; la copie :", copie.base is None)
assert transposee.base is jauges and np.shares_memory(jauges, transposee)
assert copie.base is None and not np.shares_memory(jauges, copie)
assert np.shares_memory(jauges, decoupe), "une tranche est une vue"
decoupe[0, 0] = -1.0
assert jauges[0, 0] == -1.0 and copie[0, 0] != -1.0, "écrire dans une vue écrit dans l'original"
jauges[0, 0] = copie[0, 0]

types : float64 et float32 ; octets : 4000000 et 2000000
la transposée est une vue : True ; la copie : True


## Exercice 6.2 : Réduction par axe et type `float32` (15 min)

Une campagne a mesuré $10\,000$ pièces avec $50$ jauges chacune : tirez un tableau `jauges` de forme $(10\,000, 50)$ de contraintes normales, de moyenne $100$ MPa et d'écart-type $15$ MPa (`rng.normal`, graine $2026$). Cherchez la contrainte maximale de chaque pièce, puis la moyenne de ces maxima. Convertissez ce tableau en `float32` avec `astype` et recalculez la moyenne des maxima : quelle mémoire gagne-t-on, et l'écart relatif sur la moyenne des maxima dépasse-t-il $10^{-6}$ ?

**Résultat attendu.** Maxima de forme $(10\,000{,})$ ; moyenne des maxima de 133,8 MPa ; mémoire de 4 Mo en `float64`, de 2 Mo en `float32` ; écart relatif inférieur à $10^{-6}$.

**Seconde voie.** Sur le tableau transposé `jauges.T`, le maximum selon l'axe $0$ égale le maximum selon l'axe $1$ de `jauges`.

**Indication.** Tirez `jauges` par `rng.normal`, de forme $(10\,000, 50)$. Les maxima par pièce sont `jauges.max(axis=1)`, de forme $(10\,000{,})$, et leur moyenne est `maxima.mean()`. Avec `astype`, `nbytes` passe de $4\,000\,000$ à $2\,000\,000$ octets, et la moyenne des maxima ne change qu'au septième chiffre.

Les contrôles de la cellule lisent les variables `ecart_relatif`, `maxima`, `moyenne_maxima` : gardez ces noms.

In [10]:
# Solution :
maxima = jauges.max(axis=1)
moyenne_maxima = float(maxima.mean())
moyenne_maxima_32 = float(simple.max(axis=1).mean())
ecart_relatif = abs(moyenne_maxima_32 - moyenne_maxima) / moyenne_maxima
print(f"maxima par pièce : forme {maxima.shape}, moyenne {moyenne_maxima:.2f} MPa")
print(f"mémoire : {jauges.nbytes / 1e6:.1f} Mo en float64, {simple.nbytes / 1e6:.1f} Mo en float32")
print("écart relatif sur la moyenne des maxima inférieur à 1e-6 :", ecart_relatif < 1e-6)
# Fin de la solution
assert maxima.shape == (10000,)
assert abs(moyenne_maxima - 133.76) < 0.005, "moyenne des maxima attendue : 133,76 MPa"
assert jauges.nbytes == 4_000_000 and simple.nbytes == 2_000_000, "4 Mo contre 2 Mo"
assert ecart_relatif < 1e-6, "float32 garde sept chiffres : l'écart reste sous 1e-6"
assert np.array_equal(maxima, jauges.T.max(axis=0)), "axe 1 de A égale axe 0 de la transposée"
print("Exercice 6.2 : contrôles passés")

maxima par pièce : forme (10000,), moyenne 133.76 MPa
mémoire : 4.0 Mo en float64, 2.0 Mo en float32
écart relatif sur la moyenne des maxima inférieur à 1e-6 : True
Exercice 6.2 : contrôles passés


### Section 6.3 : un tenseur à trois dimensions range une série de matrices.

In [11]:
# notion: tenseur3d
chiffres = load_digits()
imgs = chiffres.data.reshape((-1, 8, 8))            # 1 797 images de 8 x 8 pixels, niveaux de 0 à 16 (noms de l'auteur)
intensite_totale = imgs.sum(axis=(1, 2))             # une somme par image : forme (1797,)
print("images :", imgs.shape, "; intensités :", intensite_totale.shape, "; niveau maximal :", imgs.max())
assert imgs.shape == (1797, 8, 8) and intensite_totale.shape == (1797,)
assert np.array_equal(imgs, chiffres.images), "même tableau que celui de scikit-learn"
assert intensite_totale[0] == imgs[0].sum() and imgs.max() == 16
# Centroïde pondéré par l'intensité : la grille des indices se construit par diffusion, comme en 6.1 (code de l'auteur).
rows = np.arange(8).reshape(-1, 1)                   # indices de ligne, forme (8, 1)
cols = np.arange(8).reshape(1, -1)                   # indices de colonne, forme (1, 8)
num_r = (imgs * rows).sum(axis=(1, 2))               # (1797, 8, 8) fois (8, 1) : la diffusion étire rows sur chaque image
num_c = (imgs * cols).sum(axis=(1, 2))
centroide_ligne = safe_div(num_r, intensite_totale)      # division protégée : une image vide a pour centroïde 0
centroide_colonne = safe_div(num_c, intensite_totale)
print(f"centroïde de la première image : ligne {centroide_ligne[0]:.2f}, colonne {centroide_colonne[0]:.2f}")

images : (1797, 8, 8) ; intensités : (1797,) ; niveau maximal : 16.0
centroïde de la première image : ligne 3.36, colonne 3.56


In [12]:
assert centroide_ligne.shape == (1797,), "un centroïde par image"
assert 0.0 <= centroide_ligne.min() <= centroide_ligne.max() <= 7.0, "centroïde dans l'image"
assert (
    0.0 <= centroide_colonne.min() and centroide_colonne.max() <= 7.0
), "le centroïde reste dans l'image"
assert (
    abs(centroide_ligne[0] - num_r[0] / intensite_totale[0]) < 1e-12
), "même valeur par division simple"
# Symétrie gauche-droite : 1 moins l'écart à l'image retournée, rapporté à l'intensité (retournement par tranche).
imgs_flipv = imgs[:, :, ::-1]
imgs_sum = intensite_totale.copy()
imgs_sum[imgs_sum == 0] = 1.0                        # évite une division par zéro sur les images vides
sym_v = (1.0 - np.abs(imgs - imgs_flipv).sum(axis=(1, 2)) / imgs_sum).clip(0.0, 1.0)
symetrique = np.ones((1, 8, 8))

In [13]:
assert (
    abs((1.0 - np.abs(symetrique - symetrique[:, :, ::-1]).sum(axis=(1, 2)) / 64.0)[0] - 1.0) < 1e-12
), "image uniforme"
print(f"symétrie gauche-droite moyenne des chiffres : {sym_v.mean():.3f}")
assert sym_v.shape == (1797,) and 0.0 <= sym_v.min() and sym_v.max() <= 1.0
# notion: tenseur-contraintes
sigma = np.array([[120.0, 40.0, 0.0],
                  [40.0, -30.0, 0.0],
                  [0.0, 0.0, 60.0]])                 # tenseur symétrique en MPa : sxx, syy, szz et txy
principales = np.linalg.eigvalsh(sigma)              # valeurs propres d'une matrice symétrique, rangées en croissant
print("contraintes principales (MPa) :", np.round(principales, 3))
assert (
    np.allclose(principales, [-40.0, 60.0, 130.0])
), "contraintes principales attendues : -40 ; 60 ; 130 MPa"
s1, s2, s3 = principales[2], principales[1], principales[0]
von_mises = math.sqrt(0.5 * ((s1 - s2)**2 + (s2 - s3)**2 + (s3 - s1)**2))     # écrite à la main pour trois nombres

symétrie gauche-droite moyenne des chiffres : 0.275
contraintes principales (MPa) : [-40.  60. 130.]


In [14]:
composantes = math.sqrt(120.0**2 + 30.0**2 + 60.0**2 - 120.0 * (-30.0) - (-30.0) * 60.0 - 60.0 * 120.0 + 3 * 40.0**2)
print(f"von Mises par les valeurs principales : {von_mises:.2f} MPa ; par les composantes : {composantes:.2f} MPa")
assert abs(von_mises - composantes) < 1e-9, "les deux écritures de von Mises coïncident"
pression = np.zeros((3, 3))
np.fill_diagonal(pression, np.trace(sigma) / 3)      # partie sphérique : la pression moyenne sur la diagonale
deviateur = sigma - pression
assert abs(np.trace(deviateur)) < 1e-12, "le déviateur est de trace nulle"

von Mises par les valeurs principales : 147.99 MPa ; par les composantes : 147.99 MPa


## Exercice 6.3 : Contrainte de von Mises par `eigvalsh` (20 min)

Un point d'une pièce porte le tenseur $\sigma_{xx} = 120$, $\sigma_{yy} = -30$, $\sigma_{zz} = 60$, $\tau_{xy} = 40$ MPa, les autres cisaillements étant nuls. Calculez ses contraintes principales, puis sa contrainte de von Mises. Poursuivez avec 1 000 points : avec le même générateur `rng` de l'exercice 6.2, poursuivi sans le réinitialiser après le tirage des jauges, tirez `rng.normal(0, 50, size=(1000, 3, 3))` et symétrisez chaque matrice. Calculez d'un seul appel les contraintes de von Mises des 1 000 points, et vérifiez qu'une rotation de $30^\circ$ autour de $z$ ne les change pas, avec $Q$ la matrice de rotation d'angle $30^\circ$ autour de $z$ (lignes $(\cos\theta, -\sin\theta, 0)$, $(\sin\theta, \cos\theta, 0)$, $(0, 0, 1)$) : le tenseur tourné vaut $Q\,\sigma\,Q^{\top}$.

**Résultat attendu.** Contraintes principales de $-40$, 60 et 130 MPa ; von Mises de 147,99 MPa ; contraintes principales de forme $(1\,000, 3)$ et contraintes de von Mises de forme $(1\,000,)$ pour 1 000 points, maximum de von Mises de 298,0 MPa.

**Seconde voie.** À partir des composantes, $\sigma_{vM}^2 = \sigma_{xx}^2 + \sigma_{yy}^2 + \sigma_{zz}^2 - \sigma_{xx}\sigma_{yy} - \sigma_{yy}\sigma_{zz} - \sigma_{zz}\sigma_{xx} + 3\tau_{xy}^2$, d'où $\sigma_{vM} = 147{,}99$ MPa.

**Indication.** `np.linalg.eigvalsh` rend les trois valeurs, la plus petite en premier. Pour la pile, elle rend une forme $(1000, 3)$, et `mises_depuis` s'applique à la dernière dimension. La rotation $Q$ autour de $z$ se compose avec `np.einsum`, d'indices `"ij,njk,lk->nil"`, soit $Q\,\sigma\,Q^{\top}$ pour chaque point : les valeurs propres, donc von Mises, restent les mêmes.

Les contrôles de la cellule lisent les variables `champ`, `mises`, `mises_composantes`, `mises_tourne`, `prin` : gardez ces noms.

In [15]:
# Solution :
brut = rng.normal(0.0, 50.0, size=(1000, 3, 3))
champ = 0.5 * (brut + np.einsum("nij->nji", brut))                   # symétrisation : chaque matrice est symétrique
prin = np.linalg.eigvalsh(champ)                                      # forme (1000, 3), croissant

def mises_depuis(p):
    """Contrainte de von Mises à partir des trois contraintes principales (dernier axe de p)."""
    return np.sqrt(0.5 * ((p[..., 0] - p[..., 1])**2 + (p[..., 1] - p[..., 2])**2 + (p[..., 2] - p[..., 0])**2))

mises = mises_depuis(prin)
sxx, syy, szz = champ[:, 0, 0], champ[:, 1, 1], champ[:, 2, 2]
txy, tyz, tzx = champ[:, 0, 1], champ[:, 1, 2], champ[:, 2, 0]
mises_composantes = np.sqrt(sxx**2 + syy**2 + szz**2 - sxx * syy - syy * szz - szz * sxx + 3 * (txy**2 + tyz**2 + tzx**2))
theta = math.radians(30.0)
rotation = np.array([[math.cos(theta), -math.sin(theta), 0.0],
                     [math.sin(theta), math.cos(theta), 0.0],
                     [0.0, 0.0, 1.0]])
tourne = np.einsum("ij,njk,lk->nil", rotation, champ, rotation)     # Q sigma Q^T pour chacun des 1 000 points
mises_tourne = mises_depuis(np.linalg.eigvalsh(tourne))
print("principales :", prin.shape, "; von Mises maximal :", round(float(mises.max()), 1), "MPa")
print("tenseur d'essai : von Mises", round(von_mises, 2), "MPa")
# Fin de la solution
assert prin.shape == (1000, 3) and mises.shape == (1000,)
assert np.allclose(champ, champ.transpose(0, 2, 1)), "chaque tenseur est symétrique"
assert np.allclose(mises, mises_composantes), "von Mises : valeurs principales ou composantes"
assert np.allclose(mises, mises_tourne), "von Mises ne dépend pas du repère"
assert abs(von_mises - 147.99) < 0.005, "von Mises du tenseur d'essai attendu : 147,99 MPa"
assert np.all(mises >= 0.0)
print("Exercice 6.3 : contrôles passés")

principales : (1000, 3) ; von Mises maximal : 298.0 MPa
tenseur d'essai : von Mises 147.99 MPa
Exercice 6.3 : contrôles passés


### Section 6.4 : mesurer le temps avant d'optimiser.

In [16]:
# notion: perf-counter
depart = time.perf_counter()
somme = 0.0

for k in range(100000):
    somme = somme + k

duree_s = time.perf_counter() - depart
repetee = timeit.timeit("sum(range(1000))", number=200)       # 200 répétitions, durée totale en secondes
print("une durée positive :", duree_s > 0.0, "; une mesure répétée positive :", repetee > 0.0)
assert somme == 100000 * 99999 / 2 and duree_s > 0.0 and repetee > 0.0

def distances_boucles(n, pas, x0, y0):
    """Distances au point (x0, y0) d'une grille carrée de n x n nœuds, par deux boucles imbriquées."""
    d = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            d[i, j] = math.sqrt((j * pas - x0)**2 + (i * pas - y0)**2)
    return d

une durée positive : True ; une mesure répétée positive : True


In [17]:
def distances_diffusion(n, pas, x0, y0):
    """Les mêmes distances par diffusion : une colonne, une ligne, une seule expression."""
    xs = np.arange(n) * pas
    return np.sqrt((xs[np.newaxis, :] - x0)**2 + (xs.reshape(-1, 1) - y0)**2)

# notion: boucle-tableau
lent = distances_boucles(40, 1.0, 15.0, 25.0)
rapide = distances_diffusion(40, 1.0, 15.0, 25.0)
assert (
    np.allclose(lent, rapide)
), "avant de comparer des durées, les deux calculs donnent le même résultat"
print("les deux fonctions donnent la même grille de 40 x 40 :", bool(np.allclose(lent, rapide)))

les deux fonctions donnent la même grille de 40 x 40 : True


## Exercice 6.4 : Rapport de temps entre boucles et tableau (défi, 20 min)

Sur une grille de $300\times 300$ nœuds au pas de $1$ mm, avec le point $(150~;~150)$ mm, comparez les deux fonctions de la section. Vérifiez qu'elles donnent les mêmes distances, puis mesurez la meilleure de trois durées pour chacune. De combien de fois la diffusion est-elle plus rapide ?

**Résultat attendu.** Mêmes distances (`np.allclose` vrai) ; rapport de temps supérieur à 10, la valeur exacte dépendant du poste : à lire sur le vôtre.

**Seconde voie.** `np.meshgrid` donne aussi la même grille de distances, sans boucle, et `np.allclose` le confirme.

**Indication.** `timeit.repeat(lambda: distances_boucles(300, 1.0, 150.0, 150.0), number=1, repeat=3)` donne trois durées, dont on garde la plus petite ; la diffusion, bien plus brève, se mesure avec `number=20` puis se divise par $20$. Le rapport est la durée des boucles divisée par celle de la diffusion : il dépasse nettement 10.

Les contrôles de la cellule lisent les variables `SEUIL_RAPPORT`, `duree_boucles`, `duree_diffusion`, `rapport` : gardez ces noms.

In [18]:
N_GRILLE = 300
# Solution :
duree_boucles = min(timeit.repeat(lambda: distances_boucles(N_GRILLE, 1.0, 150.0, 150.0), number=1, repeat=3))
duree_diffusion = min(timeit.repeat(lambda: distances_diffusion(N_GRILLE, 1.0, 150.0, 150.0), number=20, repeat=3)) / 20
rapport = duree_boucles / duree_diffusion
SEUIL_RAPPORT = 10
print(f"grille de {N_GRILLE} x {N_GRILLE} nœuds ; la diffusion est au moins {SEUIL_RAPPORT} fois plus rapide :", rapport > SEUIL_RAPPORT)
# Fin de la solution
assert np.allclose(
    distances_boucles(N_GRILLE, 1.0, 150.0, 150.0), distances_diffusion(N_GRILLE, 1.0, 150.0, 150.0)
), "mêmes distances"
assert rapport > SEUIL_RAPPORT, "la diffusion est au moins 10 fois plus rapide que les boucles"
assert duree_diffusion < duree_boucles
print("Exercice 6.4 : contrôles passés")

grille de 300 x 300 nœuds ; la diffusion est au moins 10 fois plus rapide : True
Exercice 6.4 : contrôles passés


## Essais des fiches de fonction du chapitre

Chaque fiche du livre donne l'appel minimal d'une fonction. Les cellules suivantes rejouent ces appels, avec les mêmes lignes de code et la même sortie que la fiche : changez une valeur et relisez le résultat.

### Fiche F06-1 : Tableaux NumPy (`numpy.ndarray`)

In [19]:
import numpy as np

T = np.array([45e3, 62e3, 51e3, 85e3, 40e3])      # tractions, N
D = np.array([12e-3, 14e-3, 12e-3, 16e-3, 12e-3]) # diamètres, m
contraintes = 4 * T / (np.pi * D**2)              # contraintes, Pa

assert np.isclose(contraintes[0], 4 * 45e3 / (np.pi * 12e-3**2))
texte = " ".join(f"{x:.1f}" for x in contraintes / 1e6)
print(f"Contraintes : {texte.replace('.', ',')} MPa")
print(f"Toutes conformes : {np.all(contraintes < 500e6)}")

Contraintes : 397,9 402,8 450,9 422,8 353,7 MPa
Toutes conformes : True
